In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q ipywidgets plotly scikit-learn pandas numpy

In [3]:
!pip install -q gradio plotly pandas numpy

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("✅ All modules imported successfully!")

✅ All modules imported successfully!


In [5]:
# Set seed for reproducibility
np.random.seed(42)
n_samples = 2500

# Feature Generation
flare_intensity = np.random.uniform(1.0, 10.0, n_samples)  # C=1-3, M=3-7, X=7-10
cme_speed = np.random.uniform(300, 2800, n_samples)        # km/s
sep_proton_flux = np.random.uniform(10, 10000, n_samples)  # pfu
orbit_altitude = np.random.uniform(400, 2000, n_samples)   # LEO Altitude in km
in_saa_zone = np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25]) # South Atlantic Anomaly

# Calculated Ground Truth Target: SEU Risk Score (%)
target_seu_risk = (
    (flare_intensity * 4.2) + 
    (cme_speed / 2800.0 * 28.0) + 
    (np.log10(sep_proton_flux) * 7.5) + 
    (in_saa_zone * 22.0) + 
    np.random.normal(0, 1.5, n_samples)
)
target_seu_risk = np.clip(target_seu_risk, 0, 100)

# DataFrame Creation
df = pd.DataFrame({
    'flare_intensity': flare_intensity,
    'cme_speed': cme_speed,
    'sep_proton_flux': sep_proton_flux,
    'orbit_altitude': orbit_altitude,
    'in_saa_zone': in_saa_zone,
    'target_seu_risk': target_seu_risk
})

df.head()

,flare_intensity,cme_speed,sep_proton_flux,orbit_altitude,in_saa_zone,target_seu_risk
0,4.370861,2368.797305,3942.418848,1268.605012,0,69.810013
1,9.556429,2211.319487,4739.622237,1732.629752,0,88.208488
2,7.587945,1733.822379,8546.928458,600.142256,0,78.869565
3,6.387926,2690.117859,3406.643817,607.652745,0,80.178270
4,2.404168,801.186289,8697.800351,1290.909376,0,48.137315


In [6]:
# Define Features and Target
X = df[['flare_intensity', 'cme_speed', 'sep_proton_flux', 'orbit_altitude', 'in_saa_zone']]
y = df['target_seu_risk']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Engine
seu_model = RandomForestRegressor(n_estimators=100, random_state=42)
seu_model.fit(X_train, y_train)

# Evaluation
y_pred = seu_model.predict(X_test)
print(f"🌲 Model Training Complete!")
print(f"📊 R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"🎯 RMSE Error: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

🌲 Model Training Complete!
📊 R² Score: 0.9828
🎯 RMSE Error: 2.0519


In [7]:
def generate_seu_diagnostics(flare_val, speed_val, flux_val, alt_val, saa_val):
    # Prepare input vector
    input_data = pd.DataFrame([[flare_val, speed_val, flux_val, alt_val, saa_val]], 
                              columns=['flare_intensity', 'cme_speed', 'sep_proton_flux', 'orbit_altitude', 'in_saa_zone'])
    
    # Predict overall risk
    base_risk = seu_model.predict(input_data)[0]
    
    # Instrument Specific Breakdown
    payload_diagnostics = {
        "Optical CMOS Sensors": min(100.0, base_risk * 1.25),
        "Flight Computer Cache (SRAM)": min(100.0, base_risk * 0.90),
        "RF Transceiver Avionics": min(100.0, base_risk * 0.65)
    }
    
    # Generate Dynamic Action Plan
    if base_risk > 70:
        status = "🚨 CRITICAL THREAT"
        action = [
            "1. Trigger IMMEDIATE SAFE-MODE for Payload B (Optical Sensors).",
            "2. Initiate memory scrub cycle on Flight Computer Cache.",
            "3. Re-orient solar array angles away from coronal mass stream."
        ]
    elif base_risk > 45:
        status = "⚠️ ELEVATED RISK"
        action = [
            "1. Switch RF Transceivers to redundant backup bus.",
            "2. Increase telemetry sampling frequency to 10Hz.",
            "3. Prepare memory scrub buffer for next orbital pass."
        ]
    else:
        status = "✅ NOMINAL OPERATIONAL STATE"
        action = [
            "1. Standard background space weather telemetry monitored.",
            "2. All instruments operating within normal parameters."
        ]
        
    return base_risk, payload_diagnostics, status, action

print("✅ Diagnostic Engine ready.")

✅ Diagnostic Engine ready.


In [ ]:
import re
import gradio as gr
import numpy as np
import plotly.graph_objects as go
import tempfile
import os
from datetime import datetime

# ==========================================
# 1. SATELLITE PRESETS DATA
# ==========================================
PRESETS = {
    "Custom Configuration": {"alt": 800, "flare": "M5.0", "cme": 1400, "flux": 1500, "saa": True},
    "ISS (International Space Station)": {"alt": 420, "flare": "C9.0", "cme": 600, "flux": 400, "saa": True},
    "Hubble Space Telescope": {"alt": 535, "flare": "M1.0", "cme": 1100, "flux": 900, "saa": False},
    "Starlink LEO Constellation": {"alt": 550, "flare": "M5.0", "cme": 1500, "flux": 2200, "saa": True},
    "GOES-16 Geostationary Sat": {"alt": 2000, "flare": "X1.0", "cme": 2400, "flux": 8500, "saa": False},
}

FLARE_MAP = {"C1.0": 1.0, "C9.0": 2.5, "M1.0": 4.0, "M5.0": 6.5, "X1.0": 8.5, "X10.0": 10.0}

MAT_FACTORS = {
    "Aluminum (Standard - 6061)": 1.0,
    "Tantalum (High-Z Radiation Barrier)": 0.55,
    "Polyethylene (Hydrogen-Rich Protection)": 0.72,
}

MU_EARTH = 398600.4418  # km^3/s^2
R_EARTH = 6371.0
MISSION_START = datetime(2026, 3, 5, 0, 0, 0)


def load_preset(preset_name):
    p = PRESETS.get(preset_name, PRESETS["Custom Configuration"])
    return p["alt"], p["flare"], p["cme"], p["flux"], p["saa"]


# ==========================================
# 2. SMALL HTML WIDGET BUILDERS
# ==========================================
def _gauge(value, label, sub, color):
    value = max(0.0, min(100.0, value))
    r = 42
    circ = 2 * np.pi * r
    dash = circ * (value / 100.0)
    gap = circ - dash
    return f"""
    <div class="gauge-box">
      <svg width="104" height="104" viewBox="0 0 104 104">
        <circle cx="52" cy="52" r="{r}" fill="none" stroke="#132038" stroke-width="8"/>
        <circle cx="52" cy="52" r="{r}" fill="none" stroke="{color}" stroke-width="8"
          stroke-dasharray="{dash:.2f} {gap:.2f}" stroke-linecap="round"
          transform="rotate(-90 52 52)" style="filter: drop-shadow(0 0 6px {color});"/>
        <text x="52" y="50" text-anchor="middle" font-size="19" font-weight="700" fill="{color}"
          font-family="Orbitron, sans-serif">{value:.0f}%</text>
        <text x="52" y="65" text-anchor="middle" font-size="7.5" fill="#94a3b8"
          font-family="Share Tech Mono, monospace" letter-spacing="1">{sub}</text>
      </svg>
      <div class="gauge-label">{label}</div>
    </div>
    """


def build_top_bar(alert_level, alert_text, threat_label, threat_pct, timestamp):
    colors = {
        "critical": ("#ef4444", "🚨", "rgba(239,68,68,0.12)"),
        "warning": ("#f59e0b", "⚠️", "rgba(245,158,11,0.12)"),
        "nominal": ("#10b981", "✅", "rgba(16,185,129,0.12)"),
    }
    color, icon, bg = colors[alert_level]
    fill_pct = max(4, min(100, threat_pct))
    return f"""
    <div class="top-alert-bar" style="border-color:{color};">
      <div class="top-alert-left">
        <span class="top-alert-icon" style="color:{color};">{icon}</span>
        <span class="top-alert-title" style="color:{color};">{alert_text}</span>
        <span class="top-alert-sub">{threat_label}</span>
        <div class="threat-bar-track">
          <div class="threat-bar-fill" style="width:{fill_pct}%; background:{color};"></div>
        </div>
      </div>
      <div class="top-alert-right">
        <div class="sys-status"><span class="dot" style="background:#10b981;"></span>SYSTEM STATUS &nbsp;<b>ONLINE</b></div>
        <div class="sys-status">TELEMETRY LINK &nbsp;<b style="color:#10b981;">ACTIVE</b></div>
        <div class="sys-datetime">{timestamp}</div>
      </div>
    </div>
    """


def build_mission_health(system_health, radiation_risk, payload_status, mitigation_readiness):
    def tag(v, invert=False):
        score = (100 - v) if invert else v
        if score >= 70:
            return "NOMINAL", "#38bdf8"
        elif score >= 45:
            return "ELEVATED", "#f59e0b"
        else:
            return "HIGH", "#ef4444"

    sh_tag, sh_color = tag(system_health)
    rr_tag, rr_color = ("HIGH", "#ef4444") if radiation_risk > 70 else (("ELEVATED", "#f59e0b") if radiation_risk > 45 else ("NOMINAL", "#38bdf8"))
    ps_tag, ps_color = tag(payload_status)
    mr_tag, mr_color = ("READY", "#38bdf8") if mitigation_readiness >= 75 else ("LIMITED", "#f59e0b")

    gauges = (
        _gauge(system_health, "SYSTEM HEALTH", sh_tag, sh_color)
        + _gauge(radiation_risk, "RADIATION RISK", rr_tag, rr_color)
        + _gauge(payload_status, "PAYLOAD STATUS", ps_tag, ps_color)
        + _gauge(mitigation_readiness, "MITIGATION READINESS", mr_tag, mr_color)
    )
    return f"""
    <div class="panel">
      <div class="panel-header">🩺 MISSION HEALTH OVERVIEW</div>
      <div class="gauge-grid">{gauges}</div>
    </div>
    """


def build_space_weather(solar_wind, kp_index, particle_state, bz):
    kp_color = "#ef4444" if kp_index >= 6 else ("#f59e0b" if kp_index >= 4 else "#38bdf8")
    part_color = "#ef4444" if particle_state == "ELEVATED" else "#38bdf8"
    return f"""
    <div class="panel">
      <div class="panel-header">☀️ SPACE WEATHER SUMMARY</div>
      <div class="sw-row"><span>☀️ SOLAR WIND SPEED</span><span class="val">{solar_wind:.0f} KM/S</span></div>
      <div class="sw-row"><span>🧭 Kp INDEX</span><span class="val" style="color:{kp_color};">{kp_index} {"(HIGH)" if kp_index>=6 else "(ELEVATED)" if kp_index>=4 else "(LOW)"}</span></div>
      <div class="sw-row"><span>☄️ PARTICLE RADIATION</span><span class="val" style="color:{part_color};">{particle_state}</span></div>
      <div class="sw-row"><span>🧲 MAGNETIC FIELD (Bz)</span><span class="val" style="color:#ef4444;">{bz:.0f} nT</span></div>
    </div>
    """


def build_orbital_params(inclination, eccentricity, period_min, apogee, perigee, revolutions):
    return f"""
    <div class="panel orbit-param-bar">
      <div class="panel-header">🛰️ ORBITAL PARAMETERS</div>
      <div class="op-grid">
        <div class="op-item"><span class="op-label">INCLINATION</span><span class="op-val">{inclination:.1f}°</span></div>
        <div class="op-item"><span class="op-label">ECCENTRICITY</span><span class="op-val">{eccentricity:.5f}</span></div>
        <div class="op-item"><span class="op-label">PERIOD</span><span class="op-val">{period_min:.1f} MIN</span></div>
        <div class="op-item"><span class="op-label">APOGEE</span><span class="op-val">{apogee:.0f} KM</span></div>
        <div class="op-item"><span class="op-label">PERIGEE</span><span class="op-val">{perigee:.0f} KM</span></div>
        <div class="op-item"><span class="op-label">REVOLUTIONS</span><span class="op-val">{revolutions:,}</span></div>
      </div>
    </div>
    """


def build_mission_elapsed_static():
    # NOTE: Gradio inserts gr.HTML content via innerHTML, and <script> tags
    # injected that way are never executed by the browser (this is standard
    # DOM behavior, not a Gradio bug). To run JS we use the classic
    # "broken <img> + onerror" trick: an onerror handler attribute *does*
    # fire even when the element is injected dynamically.
    # A random id is used so a fresh timer/element pair is created if this
    # panel is ever re-rendered, and a window-level guard stops duplicate
    # intervals from stacking up on the same id.
    start_iso = MISSION_START.strftime("%Y-%m-%dT%H:%M:%SZ")
    uid = f"met_{np.random.randint(10**6, 10**7)}"
    return f"""
    <div class="panel">
      <div class="panel-header">⏱️ MISSION ELAPSED TIME</div>
      <div class="met-display" id="{uid}">00 : 00 : 00 : 00</div>
      <div class="met-labels"><span>DAYS</span><span>HRS</span><span>MIN</span><span>SEC</span></div>
      <img src="x" style="display:none" onerror="
        (function() {{
          if (window.__spectraTimers === undefined) {{ window.__spectraTimers = {{}}; }}
          if (window.__spectraTimers['{uid}']) {{ return; }}
          var start = new Date('{start_iso}').getTime();
          function pad(n) {{ return n.toString().padStart(2, '0'); }}
          function tick() {{
            var diff = Math.floor((new Date().getTime() - start) / 1000);
            var d = Math.floor(diff / 86400);
            var h = Math.floor((diff % 86400) / 3600);
            var m = Math.floor((diff % 3600) / 60);
            var s = diff % 60;
            var el = document.getElementById('{uid}');
            if (el) {{ el.textContent = pad(d) + ' : ' + pad(h) + ' : ' + pad(m) + ' : ' + pad(s); }}
          }}
          tick();
          window.__spectraTimers['{uid}'] = setInterval(tick, 1000);
        }})();
      ">
    </div>
    """



def build_quick_actions_static():
    items = [
        ("📄", "EXPORT REPORT"),
        ("🛡️", "MITIGATION PLAN"),
        ("📡", "TELEMETRY LOGS"),
        ("⚠️", "INCIDENT REPORT"),
    ]
    cells = "".join(
        f'<div class="qa-item"><div class="qa-icon">{icon}</div><div class="qa-label">{label}</div></div>'
        for icon, label in items
    )
    return f"""
    <div class="panel">
      <div class="panel-header">⚡ QUICK ACTIONS</div>
      <div class="qa-grid">{cells}</div>
    </div>
    """


def build_footer(timestamp):
    return f"""
    <div class="footer-bar">
      <div><span class="footer-label">DATA SOURCE</span><span class="footer-val">NOAA SWPC</span></div>
      <div><span class="footer-label">MODEL STATUS</span><span class="footer-val" style="color:#10b981;">OPERATIONAL</span></div>
      <div><span class="footer-label">LAST UPDATE</span><span class="footer-val">{timestamp}</span></div>
      <div><span class="footer-label">🔒 SECURE CHANNEL</span><span class="footer-val">AES-256 ENCRYPTED</span></div>
    </div>
    """


# ==========================================
# 3. ADVANCED DIAGNOSTIC & VISUAL ENGINE
# ==========================================
def process_aeroshield(alt, flare, cme, flux, saa, material, thickness):
    f_val = FLARE_MAP.get(flare, 2.5)

    # SAA Acceleration Factor
    saa_factor = 2.2 if (saa and alt <= 2000) else 1.0

    # Unshielded Base Risk Calculation
    raw_risk = ((f_val * 4.2) + (cme / 2800.0 * 25.0) + (np.log10(max(flux, 1)) * 7.5)) * saa_factor

    # Material Attenuation Factors
    attenuation = MAT_FACTORS.get(material, 1.0) * np.exp(-0.12 * thickness)

    # Attenuated Instrument Risks
    base_risk = min(100.0, max(1.0, raw_risk * attenuation))
    cmos_risk = min(100.0, base_risk * 1.25)
    sram_risk = min(100.0, base_risk * 0.90)
    rf_risk = min(100.0, base_risk * 0.65)

    now = datetime.utcnow()
    timestamp = now.strftime("%d %b %Y  %H:%M:%S UTC")

    # Status Alert Level / Top Bar
    if base_risk > 70:
        level = "critical"
        alert_text = "CRITICAL ALERT"
        threat_label = "SEVERE SPACE WEATHER THREAT DETECTED — SEU THREAT LEVEL: HIGH"
        actions = (
            "▶ [AUTO-EXECUTE] INITIATE SAFE-MODE: Payload B (CMOS Optical Sensors)\n"
            "▶ [MEM-SCRUB] SCHEDULE SCRUB CYCLE: Flight Computer Cache (SRAM)\n"
            "▶ [ATTITUDE ADJUST] Re-orient solar array vectors relative to coronal particle flux."
        )
    elif base_risk > 45:
        level = "warning"
        alert_text = "WARNING"
        threat_label = "ELEVATED RADIATION FIELD DETECTED — SEU THREAT LEVEL: MODERATE"
        actions = (
            "▶ Switch RF Transceiver bus to secondary redundant architecture.\n"
            "▶ Ramp telemetry downlink sampling rates to 10 Hz frequency.\n"
            "▶ Prepare pre-emptive cache refresh prior to next orbital pass."
        )
    else:
        level = "nominal"
        alert_text = "SYSTEM NOMINAL"
        threat_label = "BACKGROUND TELEMETRY NORMAL — SEU THREAT LEVEL: LOW"
        actions = (
            "▶ All subsystems operating within target thermal and radiation envelopes.\n"
            "▶ Standard heliophysics telemetry tracking active."
        )

    top_bar_html = build_top_bar(level, alert_text, threat_label, base_risk, timestamp)

    # ------------------------------------------
    # MISSION HEALTH / SPACE WEATHER SUMMARY
    # ------------------------------------------
    system_health = max(2.0, min(100.0, 100 - base_risk * 0.3))
    payload_status = max(2.0, min(100.0, 100 - ((cmos_risk + sram_risk) / 2) * 0.3))
    mitigation_readiness = max(10.0, min(100.0, 85 + thickness * 1.5 - (0 if material == "Aluminum (Standard - 6061)" else -6)))
    mission_health_html = build_mission_health(system_health, base_risk, payload_status, mitigation_readiness)

    solar_wind = 300 + cme * 0.11
    kp_index = int(min(9, round(f_val)))
    particle_state = "ELEVATED" if flux > 1000 else "NOMINAL"
    bz = -(base_risk / 8.0)
    space_weather_html = build_space_weather(solar_wind, kp_index, particle_state, bz)

    # ------------------------------------------
    # ORBITAL PARAMETERS
    # ------------------------------------------
    inclination = 51.6
    eccentricity = round(abs(np.sin(alt / 500.0)) * 0.0006 + 0.0001, 5)
    r_orbit = R_EARTH + alt
    period_min = 2 * np.pi * np.sqrt((r_orbit ** 3) / MU_EARTH) / 60.0
    apogee = alt + r_orbit * eccentricity
    perigee = alt - r_orbit * eccentricity
    days_elapsed = max(1, (now - MISSION_START).days)
    revolutions = int(days_elapsed * 24 * 60 / period_min)
    orbital_params_html = build_orbital_params(inclination, eccentricity, period_min, apogee, perigee, revolutions)

    footer_html = build_footer(timestamp)

    # ------------------------------------------
    # 3D SCI-FI GLOBE & ATMOSPHERE RENDER ENGINE
    # ------------------------------------------
    r_earth = 6371
    u, v = np.linspace(0, 2 * np.pi, 50), np.linspace(0, np.pi, 50)
    x_e = r_earth * np.outer(np.cos(u), np.sin(v))
    y_e = r_earth * np.outer(np.sin(u), np.sin(v))
    z_e = r_earth * np.outer(np.ones(np.size(u)), np.cos(v))

    # Atmospheric Glow Outer Layer
    r_atm = r_earth * 1.06
    x_atm = r_atm * np.outer(np.cos(u), np.sin(v))
    y_atm = r_atm * np.outer(np.sin(u), np.sin(v))
    z_atm = r_atm * np.outer(np.ones(np.size(u)), np.cos(v))

    # Satellite Trajectory Calculations
    r_orbit_plot = r_earth + alt
    t = np.linspace(0, 2 * np.pi, 150)
    x_sat = r_orbit_plot * np.cos(t)
    y_sat = r_orbit_plot * np.sin(t) * np.cos(np.radians(51.6))
    z_sat = r_orbit_plot * np.sin(t) * np.sin(np.radians(51.6))

    fig_3d = go.Figure()

    earth_colorscale = [
        [0.00, '#eef4f8'],  # south pole ice
        [0.07, '#c7dbe6'],  # polar sea ice edge
        [0.18, '#1b4f72'],  # deep southern ocean
        [0.34, '#1a6ea8'],  # ocean blue
        [0.44, '#2f8f7a'],  # coastal/shelf teal
        [0.50, '#4c8c3f'],  # land - green
        [0.56, '#7a6a3a'],  # land - highlands/brown
        [0.62, '#3f7a45'],  # land - green
        [0.72, '#1a6ea8'],  # ocean blue
        [0.86, '#1b4f72'],  # deep northern ocean
        [0.93, '#c7dbe6'],  # polar sea ice edge
        [1.00, '#eef4f8'],  # north pole ice
    ]
    fig_3d.add_trace(go.Surface(
        x=x_e, y=y_e, z=z_e, surfacecolor=z_e,
        colorscale=earth_colorscale, showscale=False, opacity=0.95, name="Earth Surface",
        lighting=dict(ambient=0.65, diffuse=0.75, specular=0.15, roughness=0.9),
        lightposition=dict(x=2000, y=1000, z=3000)
    ))

    fig_3d.add_trace(go.Surface(
        x=x_atm, y=y_atm, z=z_atm,
        colorscale=[[0, '#00f2fe'], [1, '#00f2fe']], showscale=False, opacity=0.12, name="Atmosphere Halo"
    ))

    if saa:
        saa_u = np.linspace(-np.pi / 4, np.pi / 4, 25)
        saa_v = np.linspace(np.pi / 3, 2 * np.pi / 3, 25)
        x_saa = (r_earth + 250) * np.outer(np.cos(saa_u), np.sin(saa_v))
        y_saa = (r_earth + 250) * np.outer(np.sin(saa_u), np.sin(saa_v))
        z_saa = (r_earth + 250) * np.outer(np.ones(np.size(saa_u)), np.cos(saa_v))
        fig_3d.add_trace(go.Mesh3d(
            x=x_saa.flatten(), y=y_saa.flatten(), z=z_saa.flatten(),
            color='#ef4444', opacity=0.35, name="SAA Danger Zone"
        ))

    fig_3d.add_trace(go.Scatter3d(
        x=x_sat, y=y_sat, z=z_sat,
        mode='lines', line=dict(color='#00f2fe', width=6), name="Orbital Trajectory"
    ))

    fig_3d.add_trace(go.Scatter3d(
        x=[x_sat[30]], y=[y_sat[30]], z=[z_sat[30]],
        mode='markers+text',
        marker=dict(size=9, color='#ff0055', symbol='diamond', line=dict(color='#ffffff', width=2)),
        text=["🛰️ TARGET SAT"], textposition="top center",
        name="Satellite Position"
    ))

    fig_3d.update_layout(
        paper_bgcolor="#02040a", plot_bgcolor="#02040a",
        scene=dict(
            bgcolor="#02040a", aspectmode='data',
            xaxis=dict(title='X (km)', gridcolor="#0f172a", zerolinecolor="#00f2fe", color="#38bdf8", showbackground=False),
            yaxis=dict(title='Y (km)', gridcolor="#0f172a", zerolinecolor="#00f2fe", color="#38bdf8", showbackground=False),
            zaxis=dict(title='Z (km)', gridcolor="#0f172a", zerolinecolor="#00f2fe", color="#38bdf8", showbackground=False)
        ),
        font=dict(family="Share Tech Mono, monospace", color="#38bdf8"),
        height=520, margin=dict(l=0, r=0, b=0, t=10),
        legend=dict(font=dict(color="#e2e8f0"), bgcolor="rgba(11, 19, 43, 0.7)")
    )

    # ------------------------------------------
    # 90-MINUTE CYBER TELEMETRY FEED
    # ------------------------------------------
    minutes = np.linspace(0, 90, 90)
    saa_peak = np.exp(-((minutes - 48) ** 2) / 50.0) * (1.2 if saa else 0.1)
    telemetry_risk = np.clip(base_risk * (0.6 + 0.4 * np.sin(minutes / 10.0) + saa_peak), 2.0, 100.0)

    fig_timeline = go.Figure()
    fig_timeline.add_trace(go.Scatter(
        x=minutes, y=telemetry_risk, mode='lines',
        line=dict(color='#00f2fe', width=3),
        fill='tozeroy', fillcolor='rgba(0, 242, 254, 0.08)',
        name="SEU Risk Profile"
    ))
    fig_timeline.add_hline(y=70, line_dash="dash", line_color="#ef4444", annotation_text="CRITICAL THRESHOLD (70%)", annotation_font_color="#ef4444")
    fig_timeline.add_hline(y=45, line_dash="dash", line_color="#f59e0b", annotation_text="WARNING THRESHOLD (45%)", annotation_font_color="#f59e0b")

    fig_timeline.update_layout(
        paper_bgcolor="#02040a", plot_bgcolor="#070d1e",
        xaxis=dict(title="ORBIT TIME ELAPSED (MINUTES)", gridcolor="#1e293b", color="#38bdf8", showgrid=True),
        yaxis=dict(title="PREDICTED SEU RISK (%)", range=[0, 105], gridcolor="#1e293b", color="#38bdf8", showgrid=True),
        font=dict(family="Share Tech Mono, monospace", color="#e2e8f0"),
        height=520, margin=dict(l=40, r=20, b=40, t=20)
    )

    return (
        top_bar_html,
        f"{base_risk:.1f}%",
        f"{cmos_risk:.1f}%",
        f"{sram_risk:.1f}%",
        f"{rf_risk:.1f}%",
        actions,
        fig_3d,
        fig_timeline,
        orbital_params_html,
        mission_health_html,
        space_weather_html,
        footer_html,
    )


# ==========================================
# 4. REPORT EXPORT GENERATOR
# ==========================================
def generate_incident_report(alt, flare, cme, flux, saa, material, thickness, status, base_risk, cmos_risk, sram_risk, rf_risk, actions):
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    clean_status = re.sub(r"<[^>]+>", " ", status)
    clean_status = re.sub(r"\s+", " ", clean_status).strip()
    report_content = f"""================================================================================
                    S.P.E.C.T.R.A. HELIOPHYSICS INCIDENT REPORT
================================================================================
TIMESTAMP          : {timestamp}
MISSION ID         : SPECTRA-LEO-DIAG-{np.random.randint(1000,9999)}
SECURITY CLASSIF   : UNCLASSIFIED // OPERATIONAL USE ONLY

--------------------------------------------------------------------------------
1. TELEMETRY & ENVIRONMENT PARAMETERS
--------------------------------------------------------------------------------
- Orbital Altitude         : {alt} km
- Solar Flare Classification: {flare}
- CME Stream Velocity      : {cme} km/s
- Proton Flux Density      : {flux} pfu
- SAA Crossing Active      : {saa}

--------------------------------------------------------------------------------
2. HARDWARE SHIELDING SPECIFICATIONS
--------------------------------------------------------------------------------
- Primary Material         : {material}
- Shielding Thickness      : {thickness} mm Al-equivalent

--------------------------------------------------------------------------------
3. DIAGNOSTIC VULNERABILITY ASSESSMENT
--------------------------------------------------------------------------------
- System Threat Status     : {clean_status}
- Overall System SEU Risk  : {base_risk}
- Optical CMOS Sensor Risk : {cmos_risk}
- Flight Computer SRAM Risk: {sram_risk}
- RF Transceiver Bus Risk  : {rf_risk}

--------------------------------------------------------------------------------
4. RECOMMENDED AUTOMATED COUNTERMEASURES
--------------------------------------------------------------------------------
{actions}

================================================================================
END OF REPORT // S.P.E.C.T.R.A. MISSION CONTROL AUTOMATION
================================================================================
"""
    tmp_file = os.path.join(tempfile.gettempdir(), "SPECTRA_Incident_Report.txt")
    with open(tmp_file, "w") as f:
        f.write(report_content)
    return tmp_file


# ==========================================
# 5. HIGH-TECH SCI-FI HUD STYLING (CSS)
# ==========================================
nasa_css = """
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700;900&family=Share+Tech+Mono&display=swap');

body, .gradio-container {
    background-color: #02040a !important;
    background-image:
        radial-gradient(circle at 50% 50%, rgba(14, 165, 233, 0.05) 0%, transparent 80%),
        linear-gradient(to bottom, rgba(2, 4, 10, 0.9), rgba(2, 4, 10, 0.95)) !important;
    color: #e2e8f0 !important;
    font-family: 'Share Tech Mono', monospace !important;
}

h1, h2, h3, .gr-button {
    font-family: 'Orbitron', sans-serif !important;
    letter-spacing: 1.5px !important;
}

/* Cyber Panel Glassmorphism */
.gr-box, .gr-form, .gr-panel, div[data-testid="column"] {
    background: rgba(11, 19, 43, 0.6) !important;
    border: 1px solid rgba(56, 189, 248, 0.25) !important;
    box-shadow: 0 0 15px rgba(0, 242, 254, 0.05), inset 0 0 15px rgba(0, 242, 254, 0.03) !important;
    backdrop-filter: blur(8px) !important;
    border-radius: 4px !important;
}

.gr-input, input, select, textarea {
    background-color: #060c1a !important;
    border: 1px solid #1e293b !important;
    color: #00f2fe !important;
    font-family: 'Share Tech Mono', monospace !important;
}

/* Neon Primary Action Button */
.gr-button-primary {
    background: linear-gradient(135deg, #0284c7 0%, #2563eb 100%) !important;
    border: 1px solid #38bdf8 !important;
    box-shadow: 0 0 20px rgba(56, 189, 248, 0.4) !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    transition: all 0.3s ease !important;
}

.gr-button-primary:hover {
    box-shadow: 0 0 30px rgba(56, 189, 248, 0.8) !important;
    transform: translateY(-1px);
}

.tab-nav button {
    font-family: 'Orbitron', sans-serif !important;
    font-size: 11px !important;
    color: #94a3b8 !important;
    background: transparent !important;
    border: none !important;
}

.tab-nav button.selected {
    color: #38bdf8 !important;
    border-bottom: 2px solid #38bdf8 !important;
    text-shadow: 0 0 8px rgba(56, 189, 248, 0.6) !important;
    background: rgba(15, 23, 42, 0.8) !important;
}

/* ---------- Top Alert Bar ---------- */
.top-alert-bar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    flex-wrap: wrap;
    gap: 16px;
    background: rgba(11, 19, 43, 0.6);
    border: 1px solid rgba(56, 189, 248, 0.25);
    border-left-width: 4px;
    border-radius: 4px;
    padding: 12px 18px;
    margin-bottom: 16px;
}
.top-alert-left { display:flex; align-items:center; gap:14px; flex-wrap: wrap; }
.top-alert-icon { font-size: 20px; }
.top-alert-title { font-family:'Orbitron',sans-serif; font-weight:700; letter-spacing:2px; font-size:14px; }
.top-alert-sub { color:#94a3b8; font-size:12px; }
.threat-bar-track { width:120px; height:8px; background:#132038; border-radius:4px; overflow:hidden; }
.threat-bar-fill { height:100%; border-radius:4px; }
.top-alert-right { display:flex; align-items:center; gap:18px; font-size:11px; color:#94a3b8; font-family:'Share Tech Mono',monospace; }
.sys-status .dot { display:inline-block; width:8px; height:8px; border-radius:50%; margin-right:4px; box-shadow:0 0 8px #10b981; }
.sys-status b { color:#e2e8f0; }
.sys-datetime { color:#38bdf8; text-shadow:0 0 8px rgba(56,189,248,0.5); }

/* ---------- Generic Panel ---------- */
.panel {
    background: rgba(11, 19, 43, 0.6);
    border: 1px solid rgba(56, 189, 248, 0.25);
    border-radius: 4px;
    padding: 14px 16px;
    margin-bottom: 14px;
}
.panel-header {
    font-family:'Orbitron',sans-serif;
    font-size:12px;
    letter-spacing:1.5px;
    color:#e2e8f0;
    margin-bottom:12px;
}

/* ---------- Gauges ---------- */
.gauge-grid {
    display:grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 10px;
}
.gauge-box { display:flex; flex-direction:column; align-items:center; }
.gauge-label { margin-top:4px; font-size:9px; color:#94a3b8; letter-spacing:1px; text-align:center; }

/* ---------- Space Weather Rows ---------- */
.sw-row {
    display:flex; justify-content:space-between; align-items:center;
    font-size:11.5px; color:#cbd5e1; padding:7px 0;
    border-bottom: 1px solid rgba(56,189,248,0.08);
}
.sw-row:last-child { border-bottom:none; }
.sw-row .val { color:#e2e8f0; font-weight:700; }

/* ---------- Mission Elapsed Time ---------- */
.met-display {
    font-family:'Orbitron',sans-serif;
    font-size:22px;
    letter-spacing:2px;
    color:#38bdf8;
    text-shadow:0 0 10px rgba(56,189,248,0.6);
    text-align:center;
    margin: 6px 0 2px 0;
}
.met-labels {
    display:flex; justify-content:space-around;
    font-size:9px; color:#64748b; letter-spacing:1px;
}

/* ---------- Quick Actions ---------- */
.qa-grid { display:grid; grid-template-columns: repeat(2, 1fr); gap:10px; }
.qa-item {
    background: rgba(6,12,26,0.7);
    border: 1px solid rgba(56,189,248,0.2);
    border-radius:4px;
    padding:12px 6px;
    text-align:center;
    cursor:pointer;
    transition: all 0.2s ease;
}
.qa-item:hover { border-color:#38bdf8; box-shadow:0 0 12px rgba(56,189,248,0.35); }
.qa-icon { font-size:18px; margin-bottom:4px; }
.qa-label { font-size:9px; letter-spacing:1px; color:#94a3b8; }

/* ---------- Orbital Parameter Bar ---------- */
.op-grid {
    display:grid;
    grid-template-columns: repeat(6, 1fr);
    gap: 10px;
}
.op-item { display:flex; flex-direction:column; align-items:center; }
.op-label { font-size:9px; color:#64748b; letter-spacing:1px; margin-bottom:4px; }
.op-val { font-family:'Orbitron',sans-serif; font-size:14px; color:#38bdf8; }

/* ---------- Footer Bar ---------- */
.footer-bar {
    display:flex; justify-content:space-between; flex-wrap:wrap; gap:16px;
    border-top:1px solid rgba(56,189,248,0.2);
    margin-top:18px; padding-top:14px;
    font-size:10.5px;
}
.footer-label { color:#64748b; letter-spacing:1px; margin-right:6px; }
.footer-val { color:#e2e8f0; font-weight:700; }
"""

# ==========================================
# 6. GRADIO APPLICATION LAYOUT
# ==========================================
with gr.Blocks(css=nasa_css, title="S.P.E.C.T.R.A. Mission Control") as demo:

    gr.HTML("""
        <div style="display:flex; align-items:center; gap:14px; border-bottom: 1px solid rgba(56, 189, 248, 0.3); padding-bottom: 14px; margin-bottom: 16px;">
            <svg width="44" height="44" viewBox="0 0 44 44" style="flex-shrink:0; filter: drop-shadow(0 0 6px rgba(56,189,248,0.6));">
                <circle cx="22" cy="22" r="20" fill="none" stroke="#38bdf8" stroke-width="1.4" opacity="0.55"/>
                <circle cx="22" cy="22" r="20" fill="none" stroke="#38bdf8" stroke-width="1.4"
                        stroke-dasharray="18 8" opacity="0.9">
                    <animateTransform attributeName="transform" type="rotate" from="0 22 22" to="360 22 22" dur="14s" repeatCount="indefinite"/>
                </circle>
                <circle cx="22" cy="22" r="13" fill="none" stroke="#00f2fe" stroke-width="1" opacity="0.4"/>
                <circle cx="22" cy="22" r="3.2" fill="#00f2fe" style="filter: drop-shadow(0 0 5px #00f2fe);"/>
                <line x1="22" y1="9" x2="22" y2="2" stroke="#38bdf8" stroke-width="1.6"/>
                <line x1="35" y1="22" x2="42" y2="22" stroke="#38bdf8" stroke-width="1.6"/>
                <line x1="22" y1="35" x2="22" y2="42" stroke="#38bdf8" stroke-width="1.6"/>
                <line x1="9" y1="22" x2="2" y2="22" stroke="#38bdf8" stroke-width="1.6"/>
                <circle cx="22" cy="22" r="20" fill="none" stroke="#ff0055" stroke-width="0" />
            </svg>
            <div>
                <div style="display:flex; align-items:baseline; gap:12px;">
                    <span style="background: linear-gradient(135deg, #0284c7, #2563eb); color: white; padding: 4px 10px; font-size: 11px; font-family: Orbitron; font-weight: bold; border-radius: 2px; box-shadow: 0 0 10px rgba(2, 132, 199, 0.5);">S.P.E.C.T.R.A. v1.0</span>
                    <span style="font-size: 20px; font-weight: 900; letter-spacing: 4px; font-family: Orbitron; color: #f8fafc; text-shadow: 0 0 10px rgba(56,189,248,0.35);">S.P.E.C.T.R.A.</span>
                </div>
                <div style="font-size: 10.5px; letter-spacing: 1.5px; font-family: 'Share Tech Mono', monospace; color: #64748b; margin-top: 3px;">
                    SPACE-WEATHER PAYLOAD EXPOSURES &amp; CALIBRATED TELEMETRY RISK ANALYZER
                </div>
            </div>
        </div>
    """)

    top_bar_output = gr.HTML()

    with gr.Row():
        # --- LEFT PANEL: MISSION CONTROLS & SHIELDING ---
        with gr.Column(scale=1):
            gr.Markdown("### 🛰️ Orbit & Satellite Presets")
            preset_dropdown = gr.Dropdown(choices=list(PRESETS.keys()), value="Custom Configuration", label="Load Satellite Mission Profile")

            gr.Markdown("### 🎛️ Heliophysics Telemetry")
            alt_input = gr.Slider(minimum=400, maximum=2000, value=800, step=50, label="Orbital Altitude (km)")
            flare_input = gr.Dropdown(choices=['C1.0', 'C9.0', 'M1.0', 'M5.0', 'X1.0', 'X10.0'], value='M5.0', label="Solar Flare Classification")
            cme_input = gr.Slider(minimum=300, maximum=2800, value=1400, step=100, label="CME Stream Velocity (km/s)")
            flux_input = gr.Slider(minimum=100, maximum=10000, value=1500, step=200, label="Proton Flux Density (pfu)")
            saa_input = gr.Checkbox(value=True, label="SAA Anomaly Crossing Active")

            gr.Markdown("### 🛡️ Hardware Radiation Shielding")
            mat_input = gr.Dropdown(
                choices=["Aluminum (Standard - 6061)", "Tantalum (High-Z Radiation Barrier)", "Polyethylene (Hydrogen-Rich Protection)"],
                value="Aluminum (Standard - 6061)",
                label="Shielding Material"
            )
            thick_input = gr.Slider(minimum=0, maximum=10, value=2, step=1, label="Shielding Thickness (mm)")

            btn = gr.Button("⚡ RE-CALCULATE DIAGNOSTICS", variant="primary")

        # --- CENTER PANEL: DIAGNOSTIC CONSOLE & TABS ---
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("🌐 3D Orbital Trajectory"):
                    plot_3d_output = gr.Plot(show_label=False)
                    orbital_params_output = gr.HTML()

                with gr.TabItem("📈 90-Min Orbital Telemetry Feed"):
                    plot_timeline_output = gr.Plot(show_label=False)

                with gr.TabItem("📊 Payload Vulnerability Matrix"):
                    with gr.Row():
                        risk_metric = gr.Textbox(label="System SEU Risk")
                        cmos_metric = gr.Textbox(label="Optical CMOS Risk")
                        sram_metric = gr.Textbox(label="SRAM Computer Risk")
                        rf_metric = gr.Textbox(label="RF Transceiver Risk")

                with gr.TabItem("⚡ Mitigation & Incident Export"):
                    action_output = gr.Textbox(label="Automated Countermeasures", lines=5)
                    export_btn = gr.Button("📄 EXPORT OFFICIAL NASA INCIDENT REPORT", variant="secondary")
                    report_file = gr.File(label="Downloadable Diagnostic Log")

        # --- RIGHT PANEL: MISSION HEALTH / SPACE WEATHER / QUICK ACTIONS ---
        with gr.Column(scale=1):
            mission_health_output = gr.HTML()
            space_weather_output = gr.HTML()
            gr.HTML(build_mission_elapsed_static())
            gr.HTML(build_quick_actions_static())

    footer_output = gr.HTML()

    # Interactive Preset Binding
    preset_dropdown.change(
        fn=load_preset,
        inputs=[preset_dropdown],
        outputs=[alt_input, flare_input, cme_input, flux_input, saa_input]
    )

    # Interactive Calculation Binding
    inputs = [alt_input, flare_input, cme_input, flux_input, saa_input, mat_input, thick_input]
    outputs = [
        top_bar_output, risk_metric, cmos_metric, sram_metric, rf_metric, action_output,
        plot_3d_output, plot_timeline_output, orbital_params_output,
        mission_health_output, space_weather_output, footer_output,
    ]

    btn.click(fn=process_aeroshield, inputs=inputs, outputs=outputs)
    demo.load(fn=process_aeroshield, inputs=inputs, outputs=outputs)

    # Export Report Binding
    export_btn.click(
        fn=generate_incident_report,
        inputs=[alt_input, flare_input, cme_input, flux_input, saa_input, mat_input, thick_input, top_bar_output, risk_metric, cmos_metric, sram_metric, rf_metric, action_output],
        outputs=[report_file]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_58/619115938.py:663: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=nasa_css, title="S.P.E.C.T.R.A. Mission Control") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://68e96229ede5742f26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_58/619115938.py:241: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/tmp/ipykernel_58/619115938.py:241: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/tmp/ipykernel_58/619115938.py:241: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/tmp/ipykernel_58/619115938.py:435: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/tmp/ipykern